# Colab setup (run once per session)

Mounts Drive, clones GitHub, installs deps.

**Training images — no zip needed:**

1. On **work PC** (has AWS `.env`):
   ```powershell
   python scripts/prepare_training_data.py --write-colab-manifest
   ```
2. Upload `data/manifests/colab_presigned.json` to Colab (Files panel) — ~1 MB, not a zip.
3. Run the **manifest download** cell below.

Then open a training notebook (`colab_train_stage2_screen.ipynb`, etc.).

In [ ]:
# Only secret: GitHub token for private repo (leave "" if public)
GITHUB_TOKEN = ""  # paste token here for private repo only — never commit

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/id-forensics-model")
BOOT = REPO / "scripts" / "colab_bootstrap.py"
user, repo = "ansisvaisla", "id-forensics-model"
url = (
    f"https://{GITHUB_TOKEN}@github.com/{user}/{repo}.git"
    if GITHUB_TOKEN else f"https://github.com/{user}/{repo}.git"
)

os.chdir("/content")
if REPO.is_dir() and not BOOT.is_file():
    print("Removing stale clone (missing bootstrap)...")
    subprocess.run(["rm", "-rf", str(REPO)], check=True)

if not REPO.is_dir():
    print("Cloning repo...")
    subprocess.run(["git", "clone", url, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "origin", "main"], check=False)

if not BOOT.is_file():
    raise FileNotFoundError(
        "colab_bootstrap.py not on GitHub yet. On work PC run:\n"
        "  git push origin main\n"
        "Then re-run this cell."
    )

sys.path.insert(0, str(REPO / "scripts"))
import colab_bootstrap as cb
cb.install_deps()

## Download training images (pre-signed manifest)

Upload `colab_presigned.json` from work PC to Colab via the **Files** panel (left sidebar).

Work PC command that creates it:
```powershell
python scripts/prepare_training_data.py --write-colab-manifest
```

In [ ]:
import sys, importlib
from pathlib import Path
sys.path.insert(0, "/content/id-forensics-model/scripts")
import colab_bootstrap as cb
importlib.reload(cb)

# Adjust path if you uploaded the file under a different name/location
MANIFEST = Path("/content/colab_presigned.json")

cb.download_from_colab_manifest(MANIFEST, rebuild_splits=True)